# Feature Engineering Comparison

This notebook compares model performance **before** and **after** applying feature engineering.

## New Features Introduced:
- **Interactions**: Age*AFP, Age*Albumin, Hemoglobin*Iron, AFP/Albumin Ratio
- **Binning**: Age Groups, AFP Levels
- **Log Transforms**: Normalized skewed features (AFP, Bilirubin, etc.)

In [ ]:
import sys
sys.path.append('../src')

from data_preprocessing import preprocess_pipeline
from model_training import train_all_models
from evaluation import evaluate_all_models, plot_model_comparison
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
RANDOM_STATE = 42
DATA_PATH = '../data/raw/hcc_dataset.csv'

## 1. Baseline Performance (Original Features)

In [ ]:
# Process data WITHOUT feature engineering
data_baseline = preprocess_pipeline(
    filepath=DATA_PATH,
    use_feature_engineering=False,
    random_state=RANDOM_STATE
)

print("\n--- Training Baseline Models ---")
models_baseline = train_all_models(
    data_baseline['X_train'], 
    data_baseline['y_train'],
    random_state=RANDOM_STATE
)

print("\n--- Baseline Results ---")
results_baseline = evaluate_all_models(
    models_baseline, 
    data_baseline['X_test'], 
    data_baseline['y_test']
)
results_baseline['Type'] = 'Baseline'

## 2. Enhanced Performance (New Features)

In [ ]:
# Process data WITH feature engineering
data_enhanced = preprocess_pipeline(
    filepath=DATA_PATH,
    use_feature_engineering=True,
    random_state=RANDOM_STATE
)

print("\n--- Training Enhanced Models ---")
models_enhanced = train_all_models(
    data_enhanced['X_train'], 
    data_enhanced['y_train'],
    random_state=RANDOM_STATE
)

print("\n--- Enhanced Results ---")
results_enhanced = evaluate_all_models(
    models_enhanced, 
    data_enhanced['X_test'], 
    data_enhanced['y_test']
)
results_enhanced['Type'] = 'Enhanced'

## 3. Comparison

In [ ]:
# Combine results
comparison_df = pd.concat([results_baseline, results_enhanced])

# Pivot for easier reading
pivot_df = comparison_df.pivot(index='model', columns='Type', values='f1_score')
pivot_df['Improvement'] = pivot_df['Enhanced'] - pivot_df['Baseline']

print("\nF1-Score Comparison:")
print(pivot_df.round(4))

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=comparison_df, x='model', y='f1_score', hue='Type', palette='viridis')
plt.title("Model F1-Score: Baseline vs. Feature Engineering")
plt.ylabel("F1 Score")
plt.ylim(0, 1.0)
plt.grid(axis='y', alpha=0.3)
plt.show()

## 4. Top Feature Importance (Enhanced Random Forest)

In [ ]:
rf_model = models_enhanced['Random Forest']
feature_names = data_enhanced['feature_names']

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(10, 12))
sns.barplot(data=importance_df, y='feature', x='importance', palette='mako')
plt.title("Top 20 Features Importance (After Feature Engineering)")
plt.xlabel("Importance")
plt.show()

print("Look for features like 'Interaction', 'Log', 'Ratio', 'Group' in the top list.")